# Phase 5: Explainable AI & Root Cause Analysis (설명 가능한 AI 및 근본 원인 분석)

이 단계에서는 학습된 블랙박스 머신러닝 예측 모델의 내부를 해석하기 위한 설명 가능한 AI (XAI) 기술인 **SHAP (Shapley Additive exPlanations)** 기법을 활용합니다. 설비 예측 분류 결과를 변수별 기여도로 수학적 분해하여 현장 엔지니어가 납득할 수 있는 고장 근본 원인 분석(RCA, Root Cause Analysis) 보고서를 도출합니다.

## 🎯 실습 목표
1. **학습된 모델 및 데이터 셋 바인딩**: Phase 3의 최적화된 XGBoost 분류 모델 로드
2. **SHAP TreeExplainer 구축**: 모델 특성에 최적화된 트리 기반 SHAP 계산 엔진 초기화
3. **글로벌 해석 (Global Explanation)**: Summary Plot을 이용해 설비 고장에 가장 지배적인 물리 특성 식별
4. **로컬 해석 & RCA (Local Explanation)**: 특정 고장 예측 사례를 쪼개어, 이상을 유발한 핵심 결함 요인 역추적

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

# 시각화 및 테마 설정
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# SHAP 시각화용 JS 초기화
shap.initjs()

# 모델 및 테스트 데이터 로드
try:
    model = joblib.load('./data/cmp_best_xgb_model.pkl')
    X_test = pd.read_csv('./data/cmp_X_test.csv')
    y_test = pd.read_csv('./data/cmp_y_test.csv')['failure']
    print("최적 모델 및 데이터 로드 성공!")
except FileNotFoundError:
    print("파일이 없습니다. 앞선 Phase 실습들을 먼저 수행해 주세요.")

## 1. SHAP TreeExplainer 생성 및 SHAP Value 연산
트리 앙상블 모델에 최적화된 SHAP 계산기인 TreeExplainer를 선언하여 테스트셋 전체의 기여도 행렬을 연산합니다.

In [ ]:
# TreeExplainer 초기화
explainer = shap.TreeExplainer(model)

# SHAP Value 계산 (계산 가속을 위해 테스트 셋 사용)
shap_values = explainer.shap_values(X_test)

print("SHAP Value 행렬 생성 완료, 차원:", shap_values.shape)

## 2. 글로벌 해석 (Global Explanation: Summary Plot)
전체 테스트 케이스의 SHAP 기여도 강도를 시각화하여, 설비 고장에 가장 결정적인 피처의 영향력을 분석합니다.

In [ ]:
# Summary Plot 시각화
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('CMP Sensor Global Feature Importance (SHAP)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. 로컬 해석 및 근본 원인 분석 (RCA: Local Force Plot)
실제 설비에 이상 고장(failure=1)이 발생한 단일 사례를 정밀 타격하여, 어떤 센서의 결함이 고장 예측을 결정적으로 견인했는지 물리적으로 해독합니다.

In [ ]:
# 실제 고장이 발생한 인덱스 탐색
failure_indices = np.where(y_test == 1)[0]

if len(failure_indices) > 0:
    target_idx = failure_indices[0]
    print(f"\n--- 고장 분석 대상 인덱스: {target_idx} ---")
    print("대상 센서 실측치:")
    print(X_test.iloc[target_idx])
    
    # Force Plot 생성 (HTML 기반 대시보드 시각화)
    # 빨간색이 고장 확률을 높이는 인자, 파란색이 정상 확률로 끌어내리는 인자입니다.
    force_p = shap.force_plot(
        explainer.expected_value,
        shap_values[target_idx, :],
        X_test.iloc[target_idx, :],
        matplotlib=True, # 주피터 플로팅 호환
        show=False
    )
    plt.title(f'CMP Failure Case {target_idx} Local Root Cause Analysis (SHAP)', fontsize=13)
    plt.show()
else:
    print("테스트셋 중 고장(failure=1) 사례가 발견되지 않았습니다. 데이터를 늘려보세요.")